# Semana 6 — Machine Learning · Challenge 1: datos y análisis exploratorio

**Curso:** SI3003 — Introducción a la IA (EAFIT) · Dataset: **Wine** (clasificación de 3 variedades)

**Entrada:** `data/raw/wine.csv`
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv`, `artifacts/data_contract.json`

> El conjunto de test se crea aquí pero **no se explora ni se usa** hasta la evaluación final (Challenge 3).


## Objetivos
- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga y profiling

In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"

# TODO 1: carga el archivo en un DataFrame llamado df  --> RESUELTO
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# TODO 2: reporta forma, tipos, nulos, duplicados y distribución de target  --> RESUELTO
print("Forma:", df.shape)
print("\nTipos de dato:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("\nDistribución del target:")
print(df["target"].value_counts().sort_index())
print("\nProporción del target:")
print(df["target"].value_counts(normalize=True).sort_index().round(3))
df.describe().T

**Conclusión de calidad:**

> El dataset tiene **178 filas y 14 columnas** (13 features numéricas + `target`). **No hay valores nulos ni filas duplicadas**, y todas las features son numéricas (`float64`), así que no se requiere imputación obligatoria ni codificación de categóricas. El `target` tiene 3 clases (`0`, `1`, `2`) con un leve desbalance (clase 1 ≈ 40 %, clase 0 ≈ 33 %, clase 2 ≈ 27 %); por eso el split será **estratificado**. Las escalas de las features son muy distintas (p.ej. `proline` ≈ cientos vs. `hue` ≈ 1), lo que anticipa la necesidad de **estandarizar** para modelos sensibles a la escala (KNN, regresión logística).

## 2. Split train/test

In [ ]:
# TODO 3: crea train_df (80%) y test_df (20%) estratificado  --> RESUELTO
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

In [ ]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("\nProporción target en train:")
print(train_df["target"].value_counts(normalize=True).sort_index().round(3))
print("\nProporción target en test:")
print(test_df["target"].value_counts(normalize=True).sort_index().round(3))

## 3. EDA sobre entrenamiento

Incluye: distribución de clases, distribuciones/boxplots de variables relevantes, correlaciones y tres hallazgos.

In [ ]:
# TODO 4: al menos tres visualizaciones usando SOLO train_df  --> RESUELTO
features = [col for col in train_df.columns if col != "target"]

# (1) Distribución de clases
plt.figure()
sns.countplot(x="target", data=train_df)
plt.title("Distribución de clases (train)")
plt.show()

In [ ]:
# (2) Boxplots de variables con mayor poder discriminante aparente
for col in ["flavanoids", "proline", "color_intensity", "alcohol"]:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x="target", y=col, data=train_df)
    plt.title(f"{col} por clase (train)")
    plt.show()

In [ ]:
# (3) Mapa de correlaciones entre features (solo train)
plt.figure(figsize=(10, 8))
corr = train_df[features].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlaciones entre features (train)")
plt.show()

# Pares de features mas correlacionados (|r| alto)
corr_pairs = (corr.abs().where(~np.eye(len(corr), dtype=bool))
              .stack().sort_values(ascending=False))
print("Pares más correlacionados:")
print(corr_pairs.head(6).round(3))

**Hallazgos:**

1. **`flavanoids`, `total_phenols` y `od280/od315_of_diluted_wines` separan muy bien las clases:** en los boxplots la clase 2 tiene valores claramente más bajos de flavonoides que las clases 0 y 1, lo que sugiere que serán features muy informativas para el clasificador.
2. **Las escalas son heterogéneas:** `proline` toma valores en los cientos/miles mientras que `hue` o `nonflavanoid_phenols` están por debajo de 2. Esto confirma que modelos basados en distancia (KNN) o con regularización (regresión logística) necesitan **StandardScaler**.
3. **Hay multicolinealidad moderada:** `total_phenols` y `flavanoids` están fuertemente correlacionadas (r ≈ 0.86), igual que `od280/od315` con `flavanoids`. Varias features aportan información redundante, algo que modelos como el árbol o la regresión logística regularizada manejan sin problema.

## 4. Persistencia de datos y contrato

In [ ]:
# TODO 5: reinicia índices, guarda train.csv/test.csv y data_contract.json  --> RESUELTO
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

data_contract = {
    "target": "target",
    "features": features,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "n_classes": int(df["target"].nunique()),
    "n_rows_total": int(len(df)),
}
(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(data_contract, indent=2), encoding="utf-8"
)
print(json.dumps(data_contract, indent=2))

In [ ]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

- [x] Cargué el CSV desde `data/raw`.
- [x] Revisé calidad y distribución de la clase.
- [x] Realicé un split estratificado.
- [x] No exploré el conjunto de test.
- [x] Guardé los tres artefactos requeridos.